# 🧪 Paid-API tests on a bigger OPEN Qwen (Groq)

**Separate from the production T4 notebook**, so the bot work stays clean. Two experiments, both on **Qwen3-32B** (`qwen/qwen3-32b`, Groq's hosted 32B open model):

- **Steps 1–3 — the extractor** (CPU, no re-scrape): can a larger open model rebuild the offer catalogue cleanly enough to maintain it automatically? Reuses the cached crawl + a hosted model, so **no GPU and no heavy install**, and your gold is never overwritten (drafts only).
- **Step 4 — the bot itself** on the big model: compare **accuracy + speed** vs the local 7B. Generation goes to Qwen3-32B on Groq while retrieval/voice stay local — so this part needs a **T4 GPU runtime** and the normal install.

**Open-source only** (Theme-5 brief): Qwen3-32B is open; Groq's free tier is used here for a one-time DEV comparison, not production.

> One-time: the repo is private, so add a read-only GitHub token as a Colab secret named **`GH_TOKEN`** (key icon → Add new secret → enable Notebook access).

## Step 1 — Pull the code + use the cached scrape (no re-crawl, no GPU)

In [ ]:
# Pulls the private repo (which already contains the cached scrape and the gold catalogues).
# No pip install, no torch: the Groq path uses only the standard library + a hosted model.
import os, sys, shutil, subprocess, json
from google.colab import userdata

OWNER, REPO_NAME = "YacefMehdi", "DjezzyBot"
PROJ = "/content/DjezzyBot"
CLEAN_URL = f"https://github.com/{OWNER}/{REPO_NAME}.git"
try:
    token = userdata.get("GH_TOKEN")
except Exception as e:
    raise SystemExit("Add the GH_TOKEN Colab secret (key icon ▸ add GH_TOKEN ▸ enable "
                     f"Notebook access) and re-run. ({type(e).__name__})")
auth_url = f"https://{token}@github.com/{OWNER}/{REPO_NAME}.git"

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError((r.stderr or r.stdout).replace(token, "***"))
    return r.stdout

if os.path.isdir(os.path.join(PROJ, ".git")):
    _run(f"git -C {PROJ} reset --hard -q")
    _run(f"git -C {PROJ} pull -q {auth_url} main")
else:
    if os.path.isdir(PROJ):
        shutil.rmtree(PROJ)
    _run(f"git clone -q {auth_url} {PROJ}")
    _run(f"git -C {PROJ} remote set-url origin {CLEAN_URL}")

os.chdir(PROJ)
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)

print("Code:", _run(f"git -C {PROJ} log -1 --oneline").strip())
print("Cached scrape:", len(json.load(open("data/djezzy_pages.json", encoding="utf-8"))),
      "pages (no re-crawl needed)")

## Step 2 — Extract the catalogue with the bigger open Qwen (drafts only)

The key stays in this notebook (typed via `getpass`, never stored). Get a free one at **console.groq.com ▸ API Keys**. Your gold is hashed before and after to prove the run leaves it untouched.

In [ ]:
import os, getpass, importlib, hashlib

os.environ["LLM_BACKEND"]  = "api"
os.environ["LLM_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["LLM_MODEL"]    = "qwen/qwen3-32b"    # Groq's 32B Qwen (Step 2b lists the ids)
os.environ["LLM_API_KEY"]  = getpass.getpass("Groq API key (free, console.groq.com): ")

h = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
gold_before = (h("data/offers.json"), h("data/roaming.json"))     # snapshot your gold

import build_catalog, score_catalog
importlib.reload(build_catalog); importlib.reload(score_catalog)  # re-read LLM_BACKEND=api
build_catalog.main()        # backup + *.generated.json drafts (gold not overwritten)
ok = score_catalog.main()   # accuracy vs your verified gold: MATCH / MISSING / PHANTOM

gold_after = (h("data/offers.json"), h("data/roaming.json"))
print("\nYour gold offers.json / roaming.json untouched by the extraction:",
      gold_before == gold_after)

### Step 2b (only if Step 2 says "model not found")
Lists the Qwen ids your key can use; set `LLM_MODEL` to one of them and re-run Step 2.

In [ ]:
import json, urllib.request, os, getpass
key = os.environ.get("LLM_API_KEY") or getpass.getpass("Groq API key (free, console.groq.com): ")
os.environ["LLM_API_KEY"] = key
headers = {
    "Authorization": f"Bearer {key}",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}
req = urllib.request.Request("https://api.groq.com/openai/v1/models", headers=headers)
try:
    models = [m["id"] for m in json.load(urllib.request.urlopen(req))["data"] if "qwen" in m["id"].lower()]
    print("Available Qwen models on Groq:", models)
except Exception as e:
    print("Error listing models:", e)


## Step 3 — Why the daily scraper can never corrupt the clean catalogue

The daily refresh path is **scraper → indexer → scheduler**. None of them open or write `offers.json` / `roaming.json` — only `build_catalog.py` (this test) does. The cell below proves it from the code itself.

In [ ]:
import subprocess
hits = subprocess.run(
    "grep -nE 'offers\\.json|roaming\\.json|build_catalog' scraper.py indexer.py scheduler.py",
    shell=True, capture_output=True, text=True).stdout.strip()
print("Catalogue references in the daily-refresh code (scraper / indexer / scheduler):\n")
print(hits or "  NONE → the daily scraper + indexer rebuild only the raw pages + FAISS index;\n"
               "  they never touch your curated offers.json / roaming.json.\n\n"
               "  So the nightly refresh keeps the GENERAL answers fresh, while the clean\n"
               "  catalogue is updated ONLY by this extractor — and only on demand.")

## Step 4 — Run the BOT on the big model (accuracy + speed vs the local 7B)

Launches the full assistant, but **generation goes to Qwen3-32B on Groq** while retrieval, routing, STT and TTS stay local — so it's an apples-to-apples comparison of the *generator only*. The timestamp + `t_generation` in the latency store now measure the **API speed**.

**Needs a T4 GPU runtime** (for the local embeddings/voice) and the normal install. If you ran Steps 1–3 on a CPU runtime, switch to **Runtime ▸ Change runtime type ▸ T4 GPU**, then re-run Step 1 and this cell.

In [ ]:
# Step 4 — bot on Groq. Install deps (once), point generation at the big model, launch.
import subprocess, sys
print("Installing dependencies from requirements.txt...")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

import os
os.environ["LLM_BACKEND"]  = "api"
os.environ["LLM_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["LLM_MODEL"]    = "qwen/qwen3-32b"         # same id you used in Step 2
if not os.environ.get("LLM_API_KEY"):
    import getpass; os.environ["LLM_API_KEY"] = getpass.getpass("Groq API key: ")

import app
app.main()    # generation -> Qwen3-32B on Groq; retrieval/STT/TTS stay local.
              # Ask the same questions you tried on the 7B and compare answers + the
              # response time shown in the chat. The local 7B is never loaded in this mode.